# ClaimGuard — session 5: the checks asked by the review (about 2 h 30)

1. **Oracle, done properly**: the verifier on the gold abstracts with our own sentence selection, and on the gold
   sentences, compared with the retrieved evidence **on the 188 claims that have evidence** (NEI claims cannot change).
2. **Latency** of every system with its own number of passages, on GPU, and of the full pipeline on **CPU**.
3. **Multi-hop, re-run cleanly**: retrieval pool built from the dev claims only, truncation control, the HoVer reader
   trained with 3 seeds, the full 2 x 2 end-to-end table at top 5 and top 10, the claim-only baseline.

Inputs: **claimguard-code** (upload the NEW `claimguard.zip` as a new version) **and** the session-2 output with the
verifier. GPU T4 x2, Internet on, secret HF_TOKEN (for the 7B LLM judge timing). Then *Save Version → Save & Run All*.

## 0. Setup

In [ ]:
import glob, json, os, shutil, subprocess, time, urllib.request
from pathlib import Path

candidates = [Path(p).parent for p in glob.glob("/kaggle/input/**/pyproject.toml", recursive=True)
              if "claimguard" in Path(p).read_text() and (Path(p).parent / "scripts/session5.py").exists()]
assert candidates, "claimguard.zip not found: add it as an input dataset"
SRC = candidates[0]
WORK = Path("/kaggle/working/claimguard")
if not WORK.exists():
    shutil.copytree(SRC, WORK)
os.chdir(WORK)
assert (WORK / "scripts/session5.py").exists(), "old claimguard.zip: upload the new version"
print("project copied from", SRC)

In [ ]:
!pip install -q -r requirements.txt && pip install -q -e .
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU: set the accelerator to GPU T4 x2")

In [ ]:
# run a shell command, log time and success, never stop the notebook
STATUS_FILE = WORK / "results" / "run_status_multihop.json"
STATUS_FILE.parent.mkdir(exist_ok=True)
status = []

def run(name, cmd):
    print(f"\n===== {name} =====\n$ {cmd}", flush=True)
    start = time.time()
    proc = subprocess.run(cmd, shell=True, cwd=WORK, text=True, capture_output=True)
    minutes = round((time.time() - start) / 60, 1)
    print(proc.stdout[-4000:])
    if proc.returncode != 0:
        print("STDERR:", proc.stderr[-4000:])
    status.append({"step": name, "ok": proc.returncode == 0, "minutes": minutes,
                   "error": None if proc.returncode == 0 else proc.stderr[-1500:]})
    STATUS_FILE.write_text(json.dumps(status, indent=2))
    print(f"-> {'OK' if proc.returncode == 0 else 'FAILED'} in {minutes} min", flush=True)
    return proc.returncode == 0

In [ ]:
# my SciFact verifier (session 2)
import zipfile
zips = glob.glob("/kaggle/input/**/verifier.zip", recursive=True)
dirs = [Path(p).parent for p in glob.glob("/kaggle/input/**/verifier/config.json", recursive=True)]
if zips:
    zipfile.ZipFile(zips[0]).extractall(WORK / "models")
elif dirs:
    shutil.copytree(dirs[0], WORK / "models" / "verifier", dirs_exist_ok=True)
assert (WORK / "models/verifier/config.json").exists(), "verifier not found: add the session-2 output or verifier.zip"
print("verifier OK")

In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import login
    login(token=UserSecretsClient().get_secret("HF_TOKEN"))
    print("Hugging Face login OK")
except Exception as err:
    print("no HF_TOKEN secret (only needed if the Qwen download is gated):", type(err).__name__)
STATUS_FILE = WORK / "results" / "run_status_session5.json"
run("data", "make data")

## 1. Oracle on the claims with evidence

In [ ]:
run("oracle", "python scripts/session5.py oracle --config configs/default.yaml")

## 2. Latency (GPU, then CPU only)

In [ ]:
run("latency GPU", "python scripts/session5.py latency --config configs/default.yaml --n 20")
run("latency CPU", "CUDA_VISIBLE_DEVICES= python scripts/session5.py latency --config configs/default.yaml --n 20")

## 3. HoVer and its Wikipedia

In [ ]:
HOVER = WORK / "data/raw/hover"
HOVER.mkdir(parents=True, exist_ok=True)

def fetch(url, dest, timeout=60):
    try:
        with urllib.request.urlopen(url, timeout=timeout) as r, open(dest, "wb") as f:
            shutil.copyfileobj(r, f, length=1 << 20)
        print("ok", dest.name, round(dest.stat().st_size / 1e6, 1), "MB")
        return True
    except Exception as err:
        print("FAILED", url, type(err).__name__, err)
        if dest.exists():
            dest.unlink()
        return False

RAW = "https://raw.githubusercontent.com/hover-nlp/hover/main/data/hover/"
for split in ("train", "dev"):
    fetch(RAW + f"hover_{split}_release_v1.1.json", HOVER / f"hover_{split}_release_v1.1.json")
# TF-IDF candidates of the HoVer authors (used to build the closed pool); optional
fetch("https://nlp.cs.unc.edu/data/hover/dev_tfidf_doc_retrieval_results.json",
      HOVER / "dev_tfidf_doc_retrieval_results.json", timeout=120)

In [ ]:
# the Wikipedia text: first the HoVer authors' SQLite file, else the BEIR copy of the same dump
WIKI = None
db = HOVER / "wiki_wo_links.db"
t0 = time.time()
if fetch("https://nlp.cs.unc.edu/data/hover/wiki_wo_links.db", db, timeout=300):
    WIKI = db
else:
    from huggingface_hub import hf_hub_download
    WIKI = Path(hf_hub_download("BeIR/hotpotqa", "corpus.jsonl.gz", repo_type="dataset", local_dir=str(HOVER)))
print("wiki source:", WIKI, round(WIKI.stat().st_size / 1e9, 2), "GB in", round((time.time() - t0) / 60, 1), "min")

## 4. Multi-hop, re-run

In [ ]:
run("claim-only baseline", f"python scripts/hover_claim_only.py --hover-dir {HOVER}")
run("multihop", f"python scripts/run_multihop.py --hover-dir {HOVER} --wiki {WIKI} "
    "--verifier models/verifier --train-joint --reader-seeds 13 42 2026 --out results")

## 5. Summary and download

In [ ]:
print(json.dumps(status, indent=2))
for name in ("oracle.json", "latency_cuda.json", "latency_cpu.json", "hover_claim_only.json"):
    p = WORK / "results" / name
    print("\n=====", name, "\n", p.read_text()[:3000] if p.exists() else "MISSING")
r = json.loads((WORK / "results/multihop.json").read_text())
print("\npool:", r.get("pool_kind"), r.get("pool_size"))
print("end to end:", {k: round(v["all"]["accuracy"]["value"], 3) for k, v in r["e2b_end_to_end"].items()
                      if isinstance(v, dict) and "all" in v})
out = Path("/kaggle/working/results_session5.zip")
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for p in (WORK / "results").rglob("*"):
        if p.is_file():
            z.write(p, p.relative_to(WORK))
    for log in (WORK / "models").glob("hover_joint_reader*/train_log.json"):
        z.write(log, f"results/{log.parent.name}_train_log.json")
print(out, round(out.stat().st_size / 1e6, 1), "MB")